# SCADA Wind Power – Exploratory Data Analysis

This notebook performs exploratory data analysis on the SCADA wind power dataset.

In [ ]:
import sys, os
sys.path.insert(0, os.path.join(os.getcwd(), '..'))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import yaml

sns.set_theme(style='darkgrid')
%matplotlib inline

In [ ]:
with open('../config.yaml') as f:
    config = yaml.safe_load(f)

data_path = config['data']['raw_data_path']
target_col = config['features']['target_column']
wind_col = config['features']['wind_speed_column']

print('Data path:', data_path)
print('Target column:', target_col)

In [ ]:
# Replace with your actual data path if different
# df = pd.read_csv(f'../{data_path}')
# For demonstration, generate synthetic data
np.random.seed(42)
n = 8760  # 1 year of hourly data

timestamps = pd.date_range('2022-01-01', periods=n, freq='h')
wind_speed = np.random.weibull(2, n) * 8  # Weibull distribution typical for wind
rotor_speed = wind_speed * 1.5 + np.random.normal(0, 0.5, n)
pitch_angle = np.clip(np.random.normal(5, 3, n), 0, 30)
# Approximate power curve
active_power = np.clip(0.5 * 1.225 * 7854 * wind_speed**3 / 1000, 0, 2000)
active_power += np.random.normal(0, 30, n)  # add noise

df = pd.DataFrame({
    'Datetime': timestamps,
    'Wind Speed (m/s)': wind_speed,
    'Rotor Speed (rpm)': rotor_speed,
    'Pitch Angle (deg)': pitch_angle,
    'LV ActivePower (kW)': np.clip(active_power, 0, None),
})

print(df.shape)
df.head()

In [ ]:
print('=== Dataset Info ===')
df.info()
print('\n=== Descriptive Statistics ===')
df.describe()

In [ ]:
print('Missing values per column:')
print(df.isnull().sum())

In [ ]:
numeric_cols = df.select_dtypes(include=[np.number]).columns
fig, axes = plt.subplots(len(numeric_cols), 1, figsize=(14, 4 * len(numeric_cols)))

for ax, col in zip(axes, numeric_cols):
    ax.plot(df.index[:500], df[col].values[:500], linewidth=0.8)
    ax.set_title(col)
    ax.set_ylabel(col)

plt.tight_layout()
plt.show()

In [ ]:
fig, axes = plt.subplots(1, len(numeric_cols), figsize=(5 * len(numeric_cols), 4))

for ax, col in zip(axes, numeric_cols):
    ax.hist(df[col].dropna(), bins=50, edgecolor='black', alpha=0.7)
    ax.set_title(f'Distribution: {col}')
    ax.set_xlabel(col)
    ax.set_ylabel('Count')

plt.tight_layout()
plt.show()

In [ ]:
# Power curve: Wind Speed vs Active Power
plt.figure(figsize=(10, 6))
plt.scatter(df['Wind Speed (m/s)'], df['LV ActivePower (kW)'],
            alpha=0.2, s=5, c='steelblue')
plt.xlabel('Wind Speed (m/s)')
plt.ylabel('Active Power (kW)')
plt.title('Wind Power Curve')
plt.tight_layout()
plt.show()

In [ ]:
# Correlation matrix
corr = df[numeric_cols].corr()
plt.figure(figsize=(8, 6))
sns.heatmap(corr, annot=True, cmap='coolwarm', fmt='.2f', square=True)
plt.title('Feature Correlation Matrix')
plt.tight_layout()
plt.show()

In [ ]:
# Monthly average power output
df['month'] = pd.to_datetime(df['Datetime']).dt.month
monthly_avg = df.groupby('month')['LV ActivePower (kW)'].mean()

plt.figure(figsize=(10, 5))
monthly_avg.plot(kind='bar', color='steelblue')
plt.xlabel('Month')
plt.ylabel('Average Power (kW)')
plt.title('Average Monthly Power Output')
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()